# Ejemplo 1 — Fine-tuning clásico para *sentiment analysis*

Este notebook ilustra el **fine-tuning clásico** que describes en tus slides:
**adaptar un modelo a una tarea específica** con un dataset etiquetado.

## Objetivo
Entrenar un clasificador de sentimiento a partir de un modelo preentrenado de BERT.

## Idea clave
- **Entrada**: texto, sin instrucción.
- **Salida**: etiqueta (`negative` / `positive`).
- **El modelo aprende una tarea concreta**: clasificación de sentimiento.

## Qué diferencia este notebook
En el **fine-tuning clásico**:
- el modelo se adapta a **una tarea específica**;
- el dataset tiene pares **texto → etiqueta**;
- no formulamos la tarea como instrucción en lenguaje natural.

In [ ]:
# Si hace falta, descomenta:
# !pip install -q transformers datasets evaluate accelerate scikit-learn

import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
import evaluate

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Usamos SST-2 como ejemplo sencillo y estándar
dataset = load_dataset("glue", "sst2")

# Subconjunto pequeño para que sea manejable en clase
small_train = dataset["train"].shuffle(seed=42).select(range(1000))
small_valid = dataset["validation"].shuffle(seed=42).select(range(300))

label_names = {0: "negative", 1: "positive"}

def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True)

train_ds = small_train.map(tokenize, batched=True)
valid_ds = small_valid.map(tokenize, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels)["f1"],
    }

train_ds = train_ds.remove_columns(["sentence", "idx"])
valid_ds = valid_ds.remove_columns(["sentence", "idx"])

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
)

training_args = TrainingArguments(
    output_dir="./ft-sentiment-bert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## Entrenamiento

En este punto el modelo ajusta sus pesos para la tarea concreta:
**clasificar sentimiento**.

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

## Inferencia

Ahora usamos el modelo ya ajustado sobre ejemplos nuevos.

In [ ]:
texts = [
    "I loved the movie, it was funny and moving.",
    "The film was boring, predictable and too long.",
    "The acting was okay, but the story was not interesting.",
]

inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
outputs = model(**inputs)
preds = outputs.logits.argmax(dim=-1).tolist()

for text, pred in zip(texts, preds):
    print(f"{text} -> {label_names[pred]}")

## Conclusión

Este notebook representa **fine-tuning clásico**:

- el modelo recibe **texto** como entrada;
- aprende una **única tarea**;
- la salida es una **etiqueta**.

Es el enfoque que mejor encaja con la definición de tus slides sobre *fine-tuning*.